**Experiment 11**

In [1]:
# Cell 1 — Install dependencies
!pip -q install opacus==1.4.0 tqdm pandas matplotlib torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.8/224.8 kB 9.2 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports + reproducibility
import os, math, random
from copy import deepcopy
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T

import matplotlib.pyplot as plt

RDP_ALPHAS = (
    [1.01, 1.05] +
    [1.1 + 0.1*i for i in range(0, 90)] +      # 1.1..10.0
    list(range(11, 64)) + [64, 128, 256, 512]
)

def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [3]:
# Cell 3 — Dataset: Fashion-MNIST + loaders
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.2860,), (0.3530,))
])

train_ds = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_ds  = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
print("train:", len(train_ds), "test:", len(test_ds))

100%|██████████| 26.4M/26.4M [00:02<00:00, 10.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 205kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.80MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 25.9MB/s]

train: 60000 test: 10000


In [4]:
# Cell 4 — PUBLIC set + IID clients (DISJOINT, recommended)
# Public points removed from private client datasets.

NUM_CLASSES = 10
PUBLIC_PER_CLASS = 20
SAMPLES_PER_CLIENT = 10
CLIENTS_PER_ROUND = 100

def extract_public_per_class(dataset, per_class=20, seed=0):
    rng = np.random.default_rng(seed)
    targets = np.array(dataset.targets)
    public_idx = []
    for k in range(NUM_CLASSES):
        cls_idx = np.where(targets == k)[0]
        rng.shuffle(cls_idx)
        public_idx.extend(cls_idx[:per_class].tolist())
    return sorted(public_idx)

public_idx = extract_public_per_class(train_ds, per_class=PUBLIC_PER_CLASS, seed=0)

all_train_idx = np.arange(len(train_ds))
mask = np.ones(len(train_ds), dtype=bool)
mask[np.array(public_idx)] = False
avail_idx = all_train_idx[mask]

NUM_CLIENTS = len(avail_idx) // SAMPLES_PER_CLIENT
private_needed = NUM_CLIENTS * SAMPLES_PER_CLIENT
avail_idx = avail_idx[:private_needed]

def build_iid_clients_from_indices(indices, num_clients, samples_per_client, seed=0):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(indices)
    return [perm[i*samples_per_client:(i+1)*samples_per_client].tolist()
            for i in range(num_clients)]

clients = build_iid_clients_from_indices(avail_idx, NUM_CLIENTS, SAMPLES_PER_CLIENT, seed=0)

print("public samples:", len(public_idx), f"(per_class={PUBLIC_PER_CLASS})")
print("private samples:", len(avail_idx))
print("NUM_CLIENTS:", NUM_CLIENTS, "| samples/client:", SAMPLES_PER_CLIENT, "| clients/round:", CLIENTS_PER_ROUND)
print("q =", CLIENTS_PER_ROUND / NUM_CLIENTS)

public samples: 200 (per_class=20)
private samples: 59800
NUM_CLIENTS: 5980 | samples/client: 10 | clients/round: 100
q = 0.016722408026755852


In [5]:
# Cell 6 — Model + evaluation
class FMNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64*7*7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)  # 14x14
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)  # 7x7
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / total

In [6]:
# Cell 7 — Tensor-list + flatten helpers
def model_param_list(model):
    return [p for p in model.parameters() if p.requires_grad]

@torch.no_grad()
def zero_like_params(model):
    return [torch.zeros_like(p.data) for p in model_param_list(model)]

@torch.no_grad()
def add_update_(model, update_list, scale=1.0):
    for p, u in zip(model_param_list(model), update_list):
        p.data.add_(u, alpha=scale)

@torch.no_grad()
def add_scaled_list_(dst, src, alpha):
    for d, s in zip(dst, src):
        d.add_(s, alpha=float(alpha))

@torch.no_grad()
def norm_sq_list(tlist):
    s = None
    for t in tlist:
        v = (t*t).sum()
        s = v if s is None else s + v
    return s + 1e-12

@torch.no_grad()
def l2_norm_list(tlist):
    return torch.sqrt(norm_sq_list(tlist))

def sub_list(a, b):
    return [x - y for x, y in zip(a, b)]

@torch.no_grad()
def dot_list(a_list, b_list):
    s = None
    for a, b in zip(a_list, b_list):
        v = (a*b).sum()
        s = v if s is None else s + v
    return s

@torch.no_grad()
def flatten_list(tlist):
    return torch.cat([t.reshape(-1) for t in tlist])

@torch.no_grad()
def unflatten_like(vec, template_list):
    out = []
    idx = 0
    for t in template_list:
        n = t.numel()
        out.append(vec[idx:idx+n].view_as(t))
        idx += n
    return out

In [7]:
# Cell 8 — Local client update (SGD) returns delta = (local - global)
loss_fn = nn.CrossEntropyLoss()

def client_update(global_model, client_indices, lr, momentum,
                  local_epochs=1, batch_size=10):
    local_model = deepcopy(global_model).to(device)
    local_model.train()

    loader = DataLoader(
        Subset(train_ds, client_indices),
        batch_size=batch_size, shuffle=True, drop_last=False
    )

    opt = torch.optim.SGD(local_model.parameters(), lr=lr, momentum=momentum)

    for _ in range(int(local_epochs)):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(local_model(x), y)
            loss.backward()
            opt.step()

    delta = []
    for lp, gp in zip(model_param_list(local_model), model_param_list(global_model)):
        delta.append((lp.data - gp.data).detach())
    return delta

In [8]:
# Cell 9 — DP accounting (RDP) + sigma search
from opacus.accountants.analysis import rdp as rdp_analysis

def _compute_rdp(q, noise_multiplier, steps, orders):
    try:
        return rdp_analysis.compute_rdp(q=q, noise_multiplier=noise_multiplier, steps=steps, orders=orders)
    except TypeError:
        return rdp_analysis.compute_rdp(q, noise_multiplier, steps, orders)

def _get_eps(orders, rdp, delta):
    try:
        eps, _ = rdp_analysis.get_privacy_spent(orders=orders, rdp=rdp, delta=delta)
    except TypeError:
        eps, _ = rdp_analysis.get_privacy_spent(orders, rdp, delta)
    return float(eps)

def epsilon_from_sigma_single(sigma, q, steps, delta):
    rdp = _compute_rdp(q, float(sigma), int(steps), RDP_ALPHAS)
    return _get_eps(RDP_ALPHAS, rdp, delta)

def epsilon_from_sigma_two(sig_sel, sig_rel, q, steps, delta):
    rdp1 = _compute_rdp(q, float(sig_sel), int(steps), RDP_ALPHAS)
    rdp2 = _compute_rdp(q, float(sig_rel), int(steps), RDP_ALPHAS)
    return _get_eps(RDP_ALPHAS, rdp1 + rdp2, delta)

def find_sigma_for_target_eps_single(target_eps, q, steps, delta, iters=50):
    lo, hi = 1e-4, 1.0
    while epsilon_from_sigma_single(hi, q, steps, delta) > target_eps:
        hi *= 2.0
        if hi > 1e4:
            raise RuntimeError("Could not find sigma upper bound. Reduce steps or increase eps.")
    for _ in range(iters):
        mid = 0.5*(lo+hi)
        if epsilon_from_sigma_single(mid, q, steps, delta) > target_eps:
            lo = mid
        else:
            hi = mid
    return float(hi)

def find_sigma_rel_for_target_eps_two(target_eps, q, steps, delta, sel_factor=2.0, iters=50):
    lo, hi = 1e-4, 1.0

    def E(sig_rel):
        return epsilon_from_sigma_two(sel_factor*sig_rel, sig_rel, q, steps, delta)

    while E(hi) > target_eps:
        hi *= 2.0
        if hi > 1e4:
            raise RuntimeError("Could not find sigma upper bound (two). Reduce steps or increase eps.")
    for _ in range(iters):
        mid = 0.5*(lo+hi)
        if E(mid) > target_eps:
            lo = mid
        else:
            hi = mid
    return float(hi)

In [9]:
# Cell 10 — ALIE attack helper (constant-z)

@torch.no_grad()
def alie_attack_from_honest_constz(honest_updates, byz_count, z=2.0, direction=-1.0):
    """
    mal = mu + direction * z * std (coordinate-wise)
    z is constant across Byzantine fractions.
    """
    if byz_count <= 0:
        return []
    if len(honest_updates) == 0:
        raise ValueError("No honest updates to build ALIE.")

    P = len(honest_updates[0])
    mal = []
    for j in range(P):
        stacked = torch.stack([u[j] for u in honest_updates], dim=0)
        mu = stacked.mean(dim=0)
        sd = stacked.std(dim=0, unbiased=False) + 1e-12
        mal.append(mu + float(direction) * float(z) * sd)

    mal = [t.detach() for t in mal]
    return [[t.clone() for t in mal] for _ in range(byz_count)]

In [10]:
# Cell 10.5 — Extra attacks kept in Experiment 11:
#   SF (Sign Flipping), Min-Max, Min-Sum
# Removed completely: LF, FOE

def _norm_name(s: str):
    s = str(s).upper()
    for ch in ["-", "_", " "]:
        s = s.replace(ch, "")
    return s

@torch.no_grad()
def scale_list(upd_list, factor: float):
    return [u * float(factor) for u in upd_list]

@torch.no_grad()
def mean_update_list(updates):
    if len(updates) == 0:
        return None
    P = len(updates[0])
    out = []
    for j in range(P):
        out.append(torch.stack([u[j] for u in updates], dim=0).mean(dim=0).detach())
    return out

@torch.no_grad()
def zero_updates_from_model(model, count: int):
    out = []
    base = zero_like_params(model)
    for _ in range(int(count)):
        out.append([t.clone().detach() for t in base])
    return out

@torch.no_grad()
def _perturb_direction(mu_vec, honest_mat, kind="SGN"):
    """
    perturb in {UV, STD, SGN, MU}
    """
    kind = _norm_name(kind)
    if kind == "UV":
        nrm = torch.norm(mu_vec) + 1e-12
        p = -mu_vec / nrm
    elif kind == "STD":
        sd = honest_mat.std(dim=0, unbiased=False) + 1e-12
        p = -sd
    elif kind == "SGN":
        p = -torch.sign(mu_vec)
        p[p == 0] = -1.0
    elif kind == "MU":
        p = -mu_vec
    else:
        raise ValueError(f"Unknown perturb={kind}, use UV/STD/SGN/MU")
    if float(torch.norm(p).item()) < 1e-10:
        p = torch.randn_like(mu_vec)
        p = p / (torch.norm(p) + 1e-12)
    return p.detach()

@torch.no_grad()
def _approx_diameter_lower_bound(X, iters=3):
    m = X.shape[0]
    if m < 2:
        return 0.0
    idx = 0
    best_sq = 0.0
    for _ in range(int(iters)):
        diff = X - X[idx]
        dist_sq = (diff * diff).sum(dim=1)
        j = int(torch.argmax(dist_sq).item())
        best_sq = max(best_sq, float(dist_sq[j].item()))
        idx = j
    return math.sqrt(best_sq + 1e-12)

@torch.no_grad()
def minmax_minsum_attack_vec(honest_vecs, byz_count: int,
                             kind="MINMAX",
                             perturb="SGN",
                             diam_iters=3):
    """
    Build one malicious vector m = mu + gamma * p and replicate it.
    Constraints:
      MINMAX: max_i ||m-x_i|| <= diameter(X)
      MINSUM: sum_i ||m-x_i||^2 <= max_j sum_i ||x_j-x_i||^2
    """
    if byz_count <= 0:
        return []
    if len(honest_vecs) == 0:
        return []

    kind = _norm_name(kind)
    X = torch.stack(honest_vecs, dim=0)
    m_clients = X.shape[0]

    mu = X.mean(dim=0)
    p = _perturb_direction(mu, X, kind=perturb)
    p_norm_sq = float((p * p).sum().item()) + 1e-12

    diff = (mu.unsqueeze(0) - X)

    if kind == "MINMAX":
        D = _approx_diameter_lower_bound(X, iters=diam_iters)
        D2 = float(D * D) + 1e-12

        a = p_norm_sq
        b = 2.0 * (diff @ p)
        c = (diff * diff).sum(dim=1) - float(D2)

        disc = b * b - 4.0 * a * c
        disc = torch.clamp(disc, min=0.0)
        sqrt_disc = torch.sqrt(disc + 1e-12)

        g_plus = (-b + sqrt_disc) / (2.0 * a)
        gamma = float(torch.min(g_plus).item())
        gamma = max(0.0, gamma)

    elif kind == "MINSUM":
        norms = (X * X).sum(dim=1)
        sum_norm = norms.sum()
        sum_u = X.sum(dim=0)
        dot = X @ sum_u
        S = m_clients * norms + sum_norm - 2.0 * dot
        Smax = float(S.max().item())

        sum_mu_dist = float((diff * diff).sum().item())
        rhs = max(0.0, Smax - sum_mu_dist)
        gamma = math.sqrt(rhs / (m_clients * p_norm_sq + 1e-12))

    else:
        raise ValueError("kind must be MINMAX or MINSUM")

    mal = (mu + float(gamma) * p).detach()
    return [mal.clone() for _ in range(byz_count)]

@torch.no_grad()
def minmax_minsum_attack_from_honest(honest_updates, byz_count: int,
                                     kind="MINMAX",
                                     perturb="SGN",
                                     diam_iters=3):
    if byz_count <= 0:
        return []
    if len(honest_updates) == 0:
        return []
    template = honest_updates[0]
    honest_vecs = [flatten_list(u).detach() for u in honest_updates]
    mal_vecs = minmax_minsum_attack_vec(
        honest_vecs, byz_count, kind=kind, perturb=perturb, diam_iters=diam_iters
    )
    mal_lists = [unflatten_like(v, template) for v in mal_vecs]
    return [[t.clone().detach() for t in upd] for upd in mal_lists]

In [11]:
# Cell 11 — DP-FedAvg with attack selection
# Kept attacks: ALIE, SF, MINMAX, MINSUM

def train_dp_fedavg_robust(seed, eps_total, byz_frac,
                           sigma, delta=1e-5,
                           num_clients=None, clients_per_round=CLIENTS_PER_ROUND,
                           rounds=30, local_epochs=10, batch_size=10,
                           lr0=0.125, lr_decay=0.99, momentum=0.5,
                           clip_C=1.0,
                           attack="ALIE",
                           alie_direction=-1.0, alie_z=2.0,
                           sf_gamma=1.0,
                           minmax_perturb="SGN",
                           diam_iters=3):

    seed_all(seed)
    rng = np.random.RandomState(seed)

    if num_clients is None:
        num_clients = len(clients)

    model = FMNIST_CNN().to(device)
    attack = _norm_name(attack)

    for t in tqdm(range(rounds), desc=f"DP-FedAvg atk={attack} f={int(100*byz_frac)}%", leave=True):
        lr_t = lr0 * (lr_decay ** t)
        chosen = rng.choice(num_clients, size=clients_per_round, replace=False)

        b = int(round(byz_frac * clients_per_round))
        byz_pos = set(rng.choice(clients_per_round, size=b, replace=False)) if b > 0 else set()

        honest_updates = []
        byz_cids = []

        for j, cid in enumerate(chosen):
            if j in byz_pos:
                byz_cids.append(cid)
                continue
            honest_updates.append(
                client_update(model, clients[cid], lr=lr_t, momentum=momentum,
                              local_epochs=local_epochs, batch_size=batch_size)
            )

        if b > 0:
            if attack == "SF":
                byz_updates = []
                for cid in byz_cids:
                    dlt = client_update(model, clients[cid], lr=lr_t, momentum=momentum,
                                        local_epochs=local_epochs, batch_size=batch_size)
                    byz_updates.append(scale_list(dlt, -float(sf_gamma)))

            elif attack == "ALIE":
                byz_updates = (
                    alie_attack_from_honest_constz(
                        honest_updates, b, z=float(alie_z), direction=float(alie_direction)
                    )
                    if len(honest_updates) > 0 else zero_updates_from_model(model, b)
                )

            elif attack == "MINMAX":
                byz_updates = minmax_minsum_attack_from_honest(
                    honest_updates, b, kind="MINMAX", perturb=minmax_perturb, diam_iters=diam_iters
                )
                if len(byz_updates) != b:
                    byz_updates = zero_updates_from_model(model, b)

            elif attack == "MINSUM":
                byz_updates = minmax_minsum_attack_from_honest(
                    honest_updates, b, kind="MINSUM", perturb=minmax_perturb, diam_iters=diam_iters
                )
                if len(byz_updates) != b:
                    byz_updates = zero_updates_from_model(model, b)

            else:
                raise ValueError(f"Unknown attack={attack}")
        else:
            byz_updates = []

        sum_update = zero_like_params(model)

        for upd in honest_updates + byz_updates:
            nrm = l2_norm_list(upd).item()
            scale = min(1.0, clip_C/(nrm + 1e-12))
            add_scaled_list_(sum_update, upd, scale)

        for j in range(len(sum_update)):
            sum_update[j].add_(torch.randn_like(sum_update[j]) * (sigma * clip_C))

        avg_update = [u / clients_per_round for u in sum_update]
        add_update_(model, avg_update, scale=1.0)

    return evaluate(model, test_loader)

In [12]:
# Cell 12 — AG-PTR (anchors + train_ag_ptr)
# Kept attacks: ALIE, SF, MINMAX, MINSUM

def public_anchor_update_on_subset(global_model, subset_indices, lr, momentum,
                                   public_epochs=1, pub_batch_size=64):
    local_model = deepcopy(global_model).to(device)
    local_model.train()

    loader = DataLoader(
        Subset(train_ds, subset_indices),
        batch_size=min(int(pub_batch_size), len(subset_indices)),
        shuffle=True,
        drop_last=False
    )

    opt = torch.optim.SGD(local_model.parameters(), lr=lr, momentum=momentum)

    for _ in range(int(public_epochs)):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(local_model(x), y)
            loss.backward()
            opt.step()

    delta_pub = []
    for lp, gp in zip(model_param_list(local_model), model_param_list(global_model)):
        delta_pub.append((lp.data - gp.data).detach())
    return delta_pub

def build_public_anchors(global_model, public_indices, R, lr, momentum,
                         public_epochs=1, pub_batch=20, pub_scale=0.1, seed=0):
    rng = np.random.RandomState(seed)
    anchors = []
    for _ in range(int(R)):
        k = min(int(pub_batch), len(public_indices))
        subset = rng.choice(public_indices, size=k, replace=False).tolist()

        delta_pub = public_anchor_update_on_subset(
            global_model, subset,
            lr=lr, momentum=momentum,
            public_epochs=public_epochs,
            pub_batch_size=k
        )
        anchors.append([float(pub_scale) * d for d in delta_pub])
    return anchors

def _avg_anchor(anchor_list):
    P = len(anchor_list[0])
    out = []
    for j in range(P):
        out.append(torch.stack([a[j] for a in anchor_list], dim=0).mean(dim=0).detach())
    return out

def train_ag_ptr(seed, eps_total, byz_frac,
                 sigma_sel, sigma_rel, delta=1e-5,
                 rounds=30, local_epochs=10, batch_size=10,
                 lr0=0.125, lr_decay=0.99, momentum=0.5,
                 rho=0.79, tau=60,
                 R_pub_avg=8, pub_batch=20, pub_scale=0.1, public_epochs=1,
                 attack="ALIE",
                 alie_direction=-1.0, alie_z=2.0,
                 sf_gamma=1.0,
                 minmax_perturb="SGN",
                 diam_iters=3,
                 allow_zero=False):

    seed_all(seed)
    rng = np.random.RandomState(seed)

    num_clients = len(clients)
    clients_per_round = CLIENTS_PER_ROUND

    model = FMNIST_CNN().to(device)
    zero_anchor = zero_like_params(model)

    attack = _norm_name(attack)
    accept = 0

    for t in tqdm(range(rounds), desc=f"AG-PTR atk={attack} f={int(100*byz_frac)}%", leave=True):
        lr_t = lr0 * (lr_decay ** t)
        chosen = rng.choice(num_clients, size=clients_per_round, replace=False)

        b = int(round(byz_frac * clients_per_round))
        byz_pos = set(rng.choice(clients_per_round, size=b, replace=False)) if b > 0 else set()

        anchors_pub = build_public_anchors(
            model, public_idx,
            R=R_pub_avg, lr=lr_t, momentum=momentum,
            public_epochs=public_epochs, pub_batch=pub_batch, pub_scale=pub_scale,
            seed=seed * 100000 + t
        )
        a_pub = _avg_anchor(anchors_pub)
        a_pub_norm = float(norm_sq_list(a_pub).item())

        honest_updates, honest_slots = [], []
        byz_slots, byz_cids = [], []

        for j, cid in enumerate(chosen):
            if j in byz_pos:
                byz_slots.append(j)
                byz_cids.append(cid)
                continue

            honest_updates.append(
                client_update(model, clients[cid],
                              lr=lr_t, momentum=momentum,
                              local_epochs=local_epochs, batch_size=batch_size)
            )
            honest_slots.append(j)

        if b > 0:
            if attack == "SF":
                byz_updates = []
                for cid in byz_cids:
                    dlt = client_update(model, clients[cid],
                                        lr=lr_t, momentum=momentum,
                                        local_epochs=local_epochs, batch_size=batch_size)
                    byz_updates.append(scale_list(dlt, -float(sf_gamma)))

            elif attack == "ALIE":
                byz_updates = (
                    alie_attack_from_honest_constz(
                        honest_updates, b, z=float(alie_z), direction=float(alie_direction)
                    )
                    if len(honest_updates) > 0 else zero_updates_from_model(model, b)
                )

            elif attack == "MINMAX":
                byz_updates = minmax_minsum_attack_from_honest(
                    honest_updates, b, kind="MINMAX", perturb=minmax_perturb, diam_iters=diam_iters
                )
                if len(byz_updates) != b:
                    byz_updates = zero_updates_from_model(model, b)

            elif attack == "MINSUM":
                byz_updates = minmax_minsum_attack_from_honest(
                    honest_updates, b, kind="MINSUM", perturb=minmax_perturb, diam_iters=diam_iters
                )
                if len(byz_updates) != b:
                    byz_updates = zero_updates_from_model(model, b)

            else:
                raise ValueError(f"Unknown attack={attack}")
        else:
            byz_updates = []

        deltas = [None] * clients_per_round
        for upd, j in zip(honest_updates, honest_slots):
            deltas[j] = upd
        for upd, j in zip(byz_updates, byz_slots):
            deltas[j] = upd

        for i in range(clients_per_round):
            if deltas[i] is None:
                deltas[i] = [t.clone().detach() for t in zero_anchor]

        n_pub = 0
        assign = []
        for dlt in deltas:
            d0 = float(norm_sq_list(dlt).item())
            dp = d0 + a_pub_norm - 2.0 * float(dot_list(dlt, a_pub).item())
            if dp <= d0:
                assign.append(0); n_pub += 1
            else:
                assign.append(1)

        n_zero = clients_per_round - n_pub

        noisy_pub  = float(n_pub)  + rng.normal(0.0, float(sigma_sel))
        noisy_zero = float(n_zero) + rng.normal(0.0, float(sigma_sel))

        if noisy_pub >= noisy_zero:
            r_star, noisy_winner = 0, noisy_pub
        else:
            r_star, noisy_winner = 1, noisy_zero

        if (r_star == 1) and (not allow_zero):
            continue
        if float(noisy_winner) < float(tau):
            continue

        accept += 1
        a_star = a_pub if r_star == 0 else zero_anchor

        sum_offsets = zero_like_params(model)
        for dlt, r_i in zip(deltas, assign):
            if r_i != r_star:
                continue
            offset = sub_list(dlt, a_star)
            off_norm = float(l2_norm_list(offset).item())
            scale = min(1.0, float(rho) / (off_norm + 1e-12))
            add_scaled_list_(sum_offsets, offset, scale)

        noisy_winner_capped = min(float(clients_per_round), max(0.0, float(noisy_winner)))
        m_hat = max(float(tau), noisy_winner_capped)

        for j in range(len(sum_offsets)):
            sum_offsets[j].add_(torch.randn_like(sum_offsets[j]) * (float(sigma_rel) * float(rho)))

        mean_update = [a + (so / m_hat) for a, so in zip(a_star, sum_offsets)]
        add_update_(model, mean_update, scale=1.0)

    return evaluate(model, test_loader), accept / rounds

In [13]:
# Cell 13 — FedVRDP with attack selection
# Kept attacks: ALIE, SF, MINMAX, MINSUM

@torch.no_grad()
def topk_mask_from_vec(vec, k):
    d = vec.numel()
    k = min(int(k), d)
    mask = torch.zeros(d, device=vec.device)
    if k > 0:
        idx = torch.topk(vec.abs(), k, sorted=False).indices
        mask[idx] = 1.0
    return mask

def train_fedvrdp_robust(seed, eps_total, byz_frac,
                         sigma, delta=1e-5,
                         num_clients=None, clients_per_round=CLIENTS_PER_ROUND,
                         rounds=30, local_epochs=10, batch_size=10,
                         lr0=0.125, lr_decay=0.99, momentum=0.5,
                         clip_C=1.0, k_frac=0.3,
                         attack="ALIE",
                         alie_direction=-1.0, alie_z=2.0,
                         sf_gamma=1.0,
                         minmax_perturb="SGN",
                         diam_iters=3):

    seed_all(seed)
    rng = np.random.RandomState(seed)

    if num_clients is None:
        num_clients = len(clients)

    model = FMNIST_CNN().to(device)
    attack = _norm_name(attack)

    template = zero_like_params(model)
    d = sum(t.numel() for t in template)
    k = max(1, int(k_frac * d))

    mask = torch.ones(d, device=device)

    for t in tqdm(range(rounds), desc=f"FedVRDP atk={attack} f={int(100*byz_frac)}%", leave=True):
        lr_t = lr0 * (lr_decay ** t)
        chosen = rng.choice(num_clients, size=clients_per_round, replace=False)

        b = int(round(byz_frac * clients_per_round))
        byz_pos = set(rng.choice(clients_per_round, size=b, replace=False)) if b > 0 else set()

        honest_vecs, honest_slots = [], []
        byz_slots, byz_cids = [], []

        for j, cid in enumerate(chosen):
            if j in byz_pos:
                byz_slots.append(j)
                byz_cids.append(cid)
                continue

            dlt = client_update(model, clients[cid], lr=lr_t, momentum=momentum,
                                local_epochs=local_epochs, batch_size=batch_size)
            v = flatten_list(dlt).to(device) * mask
            nrm = torch.norm(v) + 1e-12
            v = v * min(1.0, clip_C / float(nrm))

            honest_vecs.append(v.detach())
            honest_slots.append(j)

        if b > 0:
            if attack == "SF":
                byz_vecs = []
                for cid in byz_cids:
                    dlt = client_update(model, clients[cid], lr=lr_t, momentum=momentum,
                                        local_epochs=local_epochs, batch_size=batch_size)
                    v = flatten_list(dlt).to(device) * mask
                    nrm = torch.norm(v) + 1e-12
                    v = v * min(1.0, clip_C / float(nrm))
                    v = (-float(sf_gamma)) * v
                    nrm2 = torch.norm(v) + 1e-12
                    v = v * min(1.0, clip_C / float(nrm2))
                    byz_vecs.append(v.detach())

            elif attack == "ALIE":
                if len(honest_vecs) == 0:
                    byz_vecs = [torch.zeros(d, device=device) for _ in range(b)]
                else:
                    mats = torch.stack(honest_vecs, dim=0)
                    mu = mats.mean(dim=0)
                    sd = mats.std(dim=0, unbiased=False) + 1e-12
                    mal = mu + float(alie_direction) * float(alie_z) * sd
                    mal = mal * mask
                    nrm = torch.norm(mal) + 1e-12
                    mal = mal * min(1.0, clip_C / float(nrm))
                    byz_vecs = [mal.clone().detach() for _ in range(b)]

            elif attack in ("MINMAX", "MINSUM"):
                kind = "MINMAX" if attack == "MINMAX" else "MINSUM"
                mal_vecs = minmax_minsum_attack_vec(
                    honest_vecs, b, kind=kind, perturb=minmax_perturb, diam_iters=diam_iters
                )
                if len(mal_vecs) != b:
                    byz_vecs = [torch.zeros(d, device=device) for _ in range(b)]
                else:
                    byz_vecs = []
                    for mal in mal_vecs:
                        nrm = torch.norm(mal) + 1e-12
                        mal = mal * min(1.0, clip_C / float(nrm))
                        byz_vecs.append(mal.detach())
            else:
                raise ValueError(f"Unknown attack={attack}")
        else:
            byz_vecs = []

        zero_vec = torch.zeros(d, device=device)
        updates = [None] * clients_per_round
        for v, j in zip(honest_vecs, honest_slots):
            updates[j] = v
        for v, j in zip(byz_vecs, byz_slots):
            updates[j] = v
        for j in range(clients_per_round):
            if updates[j] is None:
                updates[j] = zero_vec.clone()

        sum_vec = torch.stack(updates, dim=0).sum(dim=0)
        sum_vec = sum_vec + torch.randn_like(sum_vec) * (sigma * clip_C) * mask
        avg_vec = sum_vec / clients_per_round

        avg_list = unflatten_like(avg_vec, template)
        add_update_(model, avg_list, scale=1.0)

        mask = topk_mask_from_vec(avg_vec.detach(), k)

    return evaluate(model, test_loader)

In [14]:
# Cell 13.5 — Non-DP robust baselines
# Kept aggregators: MEDIAN, TRIMMEAN, KRUM, SPARSEFED
# Removed completely: BULYAN
# Kept attacks: ALIE, SF, MINMAX, MINSUM

@torch.no_grad()
def clip_vec_to_radius(v, radius):
    nrm = torch.norm(v) + 1e-12
    return v * min(1.0, float(radius)/float(nrm))

@torch.no_grad()
def agg_median_vec(vecs):
    X = torch.stack(vecs, dim=0)
    return X.median(dim=0).values.detach()

@torch.no_grad()
def agg_trimmed_mean_vec(vecs, beta):
    X = torch.stack(vecs, dim=0)
    m = X.shape[0]
    beta = int(beta)
    beta = min(beta, (m - 1)//2)
    Xs, _ = torch.sort(X, dim=0)
    if beta > 0:
        Xs = Xs[beta:m-beta]
    return Xs.mean(dim=0).detach()

@torch.no_grad()
def _make_projection(d, k=2048, seed=0):
    rng = np.random.RandomState(seed)
    k = min(int(k), int(d))
    idx = rng.choice(d, size=k, replace=False)
    sign = rng.choice([-1.0, 1.0], size=k).astype(np.float32)
    idx_t = torch.tensor(idx, device=device, dtype=torch.long)
    sign_t = torch.tensor(sign, device=device, dtype=torch.float32)
    return idx_t, sign_t

@torch.no_grad()
def _pairwise_dist_sq(X):
    xx = (X * X).sum(dim=1, keepdim=True)
    dist = xx + xx.t() - 2.0 * (X @ X.t())
    dist.clamp_(min=0.0)
    return dist

@torch.no_grad()
def _krum_index_from_proj(Xp, f):
    m = Xp.shape[0]
    nb = m - f - 2
    if nb <= 0:
        raise ValueError(f"Krum needs m-f-2>0, got m={m}, f={f}")
    dist = _pairwise_dist_sq(Xp)
    dist.fill_diagonal_(float("inf"))
    vals, _ = torch.topk(dist, k=nb, largest=False, sorted=False, dim=1)
    scores = vals.sum(dim=1)
    return int(torch.argmin(scores).item())

@torch.no_grad()
def agg_krum_vec(vecs, f, proj_idx, proj_sign):
    X = torch.stack(vecs, dim=0)
    Xp = X[:, proj_idx] * proj_sign
    j = _krum_index_from_proj(Xp, f)
    return X[j].detach()

@torch.no_grad()
def agg_sparsefed_vec(vecs, k_frac=0.3):
    X = torch.stack(vecs, dim=0)
    mu = X.mean(dim=0)
    d = mu.numel()
    k = max(1, int(float(k_frac) * d))
    mask = topk_mask_from_vec(mu, k)
    return (mu * mask).detach()

def train_robust_baseline(seed, byz_frac,
                          rounds=30, local_epochs=10, batch_size=10,
                          lr0=0.125, lr_decay=0.99, momentum=0.5,
                          agg="MEDIAN",
                          attack="ALIE",
                          clip_C=1.0,
                          sparse_k_frac=0.3,
                          proj_k=2048,
                          alie_direction=-1.0, alie_z=2.0,
                          sf_gamma=1.0,
                          minmax_perturb="SGN",
                          diam_iters=3):
    """
    Non-DP baselines evaluated across f.
    Returns final accuracy or NaN if the aggregator is invalid for this regime.
    """
    seed_all(seed)
    rng = np.random.RandomState(seed)

    num_clients = len(clients)
    m = CLIENTS_PER_ROUND

    model = FMNIST_CNN().to(device)
    template = zero_like_params(model)
    d = sum(t.numel() for t in template)

    proj_idx, proj_sign = _make_projection(d, k=proj_k, seed=seed + 999)

    agg = _norm_name(agg)
    attack = _norm_name(attack)

    for t in tqdm(range(rounds), desc=f"{agg}(non-DP) atk={attack} f={int(100*byz_frac)}%", leave=True):
        lr_t = lr0 * (lr_decay ** t)
        chosen = rng.choice(num_clients, size=m, replace=False)

        b = int(round(byz_frac * m))
        byz_pos = set(rng.choice(m, size=b, replace=False)) if b > 0 else set()

        if agg == "KRUM":
            if (2*b + 2) >= m:
                return float("nan")

        honest_updates, honest_slots = [], []
        byz_cids, byz_slots = [], []

        for j, cid in enumerate(chosen):
            if j in byz_pos:
                byz_slots.append(j)
                byz_cids.append(cid)
                continue
            honest_updates.append(
                client_update(model, clients[cid], lr=lr_t, momentum=momentum,
                              local_epochs=local_epochs, batch_size=batch_size)
            )
            honest_slots.append(j)

        if b > 0:
            if attack == "SF":
                byz_updates = []
                for cid in byz_cids:
                    dlt = client_update(model, clients[cid], lr=lr_t, momentum=momentum,
                                        local_epochs=local_epochs, batch_size=batch_size)
                    byz_updates.append(scale_list(dlt, -float(sf_gamma)))

            elif attack == "ALIE":
                byz_updates = (
                    alie_attack_from_honest_constz(
                        honest_updates, b, z=float(alie_z), direction=float(alie_direction)
                    )
                    if len(honest_updates) > 0 else zero_updates_from_model(model, b)
                )

            elif attack == "MINMAX":
                byz_updates = minmax_minsum_attack_from_honest(
                    honest_updates, b, kind="MINMAX", perturb=minmax_perturb, diam_iters=diam_iters
                )
                if len(byz_updates) != b:
                    byz_updates = zero_updates_from_model(model, b)

            elif attack == "MINSUM":
                byz_updates = minmax_minsum_attack_from_honest(
                    honest_updates, b, kind="MINSUM", perturb=minmax_perturb, diam_iters=diam_iters
                )
                if len(byz_updates) != b:
                    byz_updates = zero_updates_from_model(model, b)

            else:
                raise ValueError(f"Unknown attack={attack}")
        else:
            byz_updates = []

        deltas = [None] * m
        for upd, j in zip(honest_updates, honest_slots):
            deltas[j] = upd
        for upd, j in zip(byz_updates, byz_slots):
            deltas[j] = upd

        zero_upd = zero_like_params(model)
        for j in range(m):
            if deltas[j] is None:
                deltas[j] = [t.clone().detach() for t in zero_upd]

        vecs = []
        for upd in deltas:
            v = flatten_list(upd).detach()
            v = clip_vec_to_radius(v, clip_C)
            vecs.append(v)

        if agg == "MEDIAN":
            agg_vec = agg_median_vec(vecs)
        elif agg in ("TRIMMEAN", "TRIMMEDMEAN"):
            agg_vec = agg_trimmed_mean_vec(vecs, beta=b)
        elif agg == "KRUM":
            agg_vec = agg_krum_vec(vecs, f=b, proj_idx=proj_idx, proj_sign=proj_sign)
        elif agg == "SPARSEFED":
            agg_vec = agg_sparsefed_vec(vecs, k_frac=sparse_k_frac)
        else:
            raise ValueError(f"Unknown agg={agg}")

        add_update_(model, unflatten_like(agg_vec, template), scale=1.0)

    return evaluate(model, test_loader)

In [16]:
# Cell 14 — Smoke test (ALL kept attacks × ALL kept methods)
# Output: table of accuracies for each attack

import numpy as np
import pandas as pd

seed = 0
eps_total = 2.0
delta = 1e-5

ATTACKS = ["ALIE", "SF", "MINMAX", "MINSUM"]
f = 0

ROUNDS = 5
LOCAL_EPOCHS = 1
BATCH_SIZE = 10

LR0 = 0.125
LR_DECAY = 0.99
MOMENTUM = 0.5

ALIE_Z = 2.0
ALIE_DIR = -1.0
SF_GAMMA = 1.0
MINMAX_PERT = "SGN"
DIAM_ITERS = 3

RHO = 0.79
TAU = 55
R_PUB_AVG = 8
PUB_BATCH = 20
PUB_SCALE = 0.1
PUBLIC_EPOCHS = 1

CLIP_C_DP = 1.0
CLIP_C_VR = 0.5
K_FRAC = 0.30

ROBUST_CLIP_C = 1.0
SPARSE_K_FRAC = 0.30
ROBUST_AGGS = {
    "Median (non-DP)": "MEDIAN",
    "TrimmedMean (non-DP)": "TRIMMEAN",
    "Krum (non-DP)": "KRUM",
    "SparseFed (non-DP)": "SPARSEFED",
}

required = [
    "train_dp_fedavg_robust",
    "train_fedvrdp_robust",
    "train_ag_ptr",
    "train_robust_baseline",
    "find_sigma_for_target_eps_single",
    "find_sigma_rel_for_target_eps_two",
]
for name in required:
    if name not in globals():
        raise NameError(f"Missing {name}. Run the cell that defines it BEFORE Cell 14.")

q = CLIENTS_PER_ROUND / len(clients)

sigma_dp = find_sigma_for_target_eps_single(eps_total, q, ROUNDS, delta)
sel_factor = 2.0
sigma_rel = find_sigma_rel_for_target_eps_two(eps_total, q, ROUNDS, delta, sel_factor=sel_factor)
sigma_sel = sel_factor * sigma_rel

print("SMOKE settings")
print("  f =", f, "| rounds =", ROUNDS, "| local_epochs =", LOCAL_EPOCHS)
print("  attacks =", ATTACKS)
print("  sigma_dp =", sigma_dp)
print("  sigma_sel/sigma_rel =", sigma_sel, sigma_rel)

rows = []

for ATTACK in ATTACKS:
    print("\n" + "="*60)
    print(f"ATTACK: {ATTACK} | f: {f}")
    print("="*60)

    acc_dp = train_dp_fedavg_robust(
        seed, eps_total, f, sigma_dp, delta=delta,
        rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
        lr0=LR0, lr_decay=LR_DECAY, momentum=MOMENTUM,
        clip_C=CLIP_C_DP,
        attack=ATTACK,
        alie_direction=ALIE_DIR, alie_z=ALIE_Z,
        sf_gamma=SF_GAMMA,
        minmax_perturb=MINMAX_PERT, diam_iters=DIAM_ITERS
    )

    acc_vr = train_fedvrdp_robust(
        seed, eps_total, f, sigma_dp, delta=delta,
        rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
        lr0=LR0, lr_decay=LR_DECAY, momentum=MOMENTUM,
        clip_C=CLIP_C_VR, k_frac=K_FRAC,
        attack=ATTACK,
        alie_direction=ALIE_DIR, alie_z=ALIE_Z,
        sf_gamma=SF_GAMMA,
        minmax_perturb=MINMAX_PERT, diam_iters=DIAM_ITERS
    )

    acc_ag, ar_ag = train_ag_ptr(
        seed, eps_total, f,
        sigma_sel, sigma_rel, delta=delta,
        rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
        lr0=LR0, lr_decay=LR_DECAY, momentum=MOMENTUM,
        rho=RHO, tau=TAU,
        R_pub_avg=R_PUB_AVG, pub_batch=PUB_BATCH, pub_scale=PUB_SCALE, public_epochs=PUBLIC_EPOCHS,
        attack=ATTACK,
        alie_direction=ALIE_DIR, alie_z=ALIE_Z,
        sf_gamma=SF_GAMMA,
        minmax_perturb=MINMAX_PERT, diam_iters=DIAM_ITERS,
        allow_zero=False
    )

    print("DP-FedAvg acc:", acc_dp)
    print("FedVRDP   acc:", acc_vr)
    print("AG-PTR    acc:", acc_ag, "| accept_rate:", ar_ag)

    baseline_results = {}
    for name, agg in ROBUST_AGGS.items():
        try:
            acc_rb = train_robust_baseline(
                seed, f,
                rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
                lr0=LR0, lr_decay=LR_DECAY, momentum=MOMENTUM,
                agg=agg, attack=ATTACK,
                clip_C=ROBUST_CLIP_C,
                sparse_k_frac=SPARSE_K_FRAC,
                alie_direction=ALIE_DIR, alie_z=ALIE_Z,
                sf_gamma=SF_GAMMA,
                minmax_perturb=MINMAX_PERT, diam_iters=DIAM_ITERS,
                proj_k=2048
            )
            baseline_results[name] = float(acc_rb)
        except Exception as e:
            print(f"{name}: ERROR -> {repr(e)}")
            baseline_results[name] = float("nan")

        if baseline_results[name] != baseline_results[name]:
            print(f"{name}: NaN (invalid regime for this f / m={CLIENTS_PER_ROUND})")
        else:
            print(f"{name} acc:", baseline_results[name])

    row = {
        "attack": ATTACK,
        "f": f,
        "DP-FedAvg": float(acc_dp),
        "FedVRDP": float(acc_vr),
        "AG-PTR": float(acc_ag),
        "AG-PTR_accept_rate": float(ar_ag),
    }
    row.update(baseline_results)
    rows.append(row)

df_smoke_all = pd.DataFrame(rows)
cols = ["attack", "f", "DP-FedAvg", "FedVRDP", "AG-PTR", "AG-PTR_accept_rate"] + list(ROBUST_AGGS.keys())
df_smoke_all = df_smoke_all[cols]

print("\n\n===== SMOKE TABLE (ALL kept attacks) =====")
display(df_smoke_all)

df_smoke_all.to_csv("exp11_smoke_all_attacks.csv", index=False)
print("Saved exp11_smoke_all_attacks.csv")

SMOKE settings
  f = 0 | rounds = 5 | local_epochs = 1
  attacks = ['ALIE', 'SF', 'MINMAX', 'MINSUM']
  sigma_dp = 0.804078106574686
  sigma_sel/sigma_rel = 1.6087871355020584 0.8043935677510292

ATTACK: ALIE | f: 0


AG-PTR atk=ALIE f=0%: 100%|██████████| 5/5 [00:01<00:00,  4.60it/s]


DP-FedAvg acc: 0.5124
FedVRDP   acc: 0.3283
AG-PTR    acc: 0.4717 | accept_rate: 1.0


MEDIAN(non-DP) atk=ALIE f=0%: 100%|██████████| 5/5 [00:00<00:00,  5.99it/s]


Median (non-DP) acc: 0.4601


...

Krum (non-DP) acc: 0.2889


SPARSEFED(non-DP) atk=MINSUM f=0%: 100%|██████████| 5/5 [00:00<00:00,  6.01it/s]


SparseFed (non-DP) acc: 0.4964


===== SMOKE TABLE (ALL kept attacks) =====


,attack,f,DP-FedAvg,FedVRDP,AG-PTR,AG-PTR_accept_rate,Median (non-DP),TrimmedMean (non-DP),Krum (non-DP),SparseFed (non-DP)
0,ALIE,0,0.5124,0.3283,0.4717,1.0,0.4601,0.5175,0.2889,0.4964
1,SF,0,0.5124,0.3283,0.4717,1.0,0.4601,0.5175,0.2889,0.4964
2,MINMAX,0,0.5124,0.3283,0.4717,1.0,0.4601,0.5175,0.2889,0.4964
3,MINSUM,0,0.5124,0.3283,0.4717,1.0,0.4601,0.5175,0.2889,0.4964


Saved exp11_smoke_all_attacks.csv


In [15]:
# Cell 15 — FULL Experiment 11 for ALL kept attacks
# Runs attacks: ALIE, SF, MINMAX, MINSUM
# Produces:
#   1) per-attack seedwise CSV
#   2) per-attack summary CSV
#   3) combined seedwise CSV
#   4) combined summary CSV
#   5) one accuracy-vs-Byzantine plot per attack
#   6) one combined table for all attacks

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------
# SETTINGS
# -----------------------
seed_list = [0, 1, 2]
eps_total = 2.0
delta = 1e-5

ATTACKS = ["ALIE", "SF", "MINMAX", "MINSUM"]

f_list = [0.00, 0.10, 0.20, 0.30, 0.40, 0.49, 0.60]

ROUNDS = 180
LOCAL_EPOCHS = 10
BATCH_SIZE = 10

LR0 = 0.125
LR_DECAY = 0.99
MOMENTUM = 0.5

ALIE_Z = 2.0
ALIE_DIR = -1.0
SF_GAMMA = 1.0
MINMAX_PERT = "SGN"
DIAM_ITERS = 3

RHO = 0.79
TAU = 55
R_PUB_AVG = 8
PUB_BATCH = 20
PUB_SCALE = 0.1
PUBLIC_EPOCHS = 1

CLIP_C_DP = 1.0
CLIP_C_VR = 0.5
K_FRAC = 0.30

INCLUDE_NONDP = True
ROBUST_CLIP_C = 1.0
SPARSE_K_FRAC = 0.30

ROBUST_METHODS = {
    "median_nondp_acc":   ("MEDIAN",   "Median (non-DP)"),
    "trimmean_nondp_acc": ("TRIMMEAN", "TrimmedMean (non-DP)"),
    "krum_nondp_acc":     ("KRUM",     "Krum (non-DP)"),
    "sparsefed_nondp_acc":("SPARSEFED","SparseFed (non-DP)"),
}

METHOD_LABELS = {
    "dp_fedavg_acc": "DP-FedAvg",
    "fedvrdp_acc": "FedVRDP",
    "agptr_acc": "AG-PTR",
    "median_nondp_acc": "Median (non-DP)",
    "trimmean_nondp_acc": "TrimmedMean (non-DP)",
    "krum_nondp_acc": "Krum (non-DP)",
    "sparsefed_nondp_acc": "SparseFed (non-DP)",
}

# -----------------------
# SANITY CHECKS
# -----------------------
required = [
    "train_dp_fedavg_robust",
    "train_fedvrdp_robust",
    "train_ag_ptr",
    "train_robust_baseline",
    "find_sigma_for_target_eps_single",
    "find_sigma_rel_for_target_eps_two",
]
for name in required:
    if name not in globals():
        raise NameError(f"Missing {name}. Run the cell that defines it before running Cell 15.")

# -----------------------
# PRIVACY SIGMAS (once)
# -----------------------
q = CLIENTS_PER_ROUND / len(clients)

sigma_dp = find_sigma_for_target_eps_single(eps_total, q, ROUNDS, delta)
eps_dp = epsilon_from_sigma_single(sigma_dp, q, ROUNDS, delta)

sel_factor = 2.0
sigma_rel = find_sigma_rel_for_target_eps_two(eps_total, q, ROUNDS, delta, sel_factor=sel_factor)
sigma_sel = sel_factor * sigma_rel
eps_ag = epsilon_from_sigma_two(sigma_sel, sigma_rel, q, ROUNDS, delta)

print("FULL Exp11 settings")
print("  attacks:", ATTACKS)
print("  eps_total:", eps_total, "delta:", delta)
print("  rounds:", ROUNDS, "local_epochs:", LOCAL_EPOCHS, "batch:", BATCH_SIZE)
print("  DP sigma:", sigma_dp, "achieved eps≈", eps_dp)
print("  AG sigmas sel/rel:", sigma_sel, sigma_rel, "achieved eps≈", eps_ag)
print("  INCLUDE_NONDP:", INCLUDE_NONDP)

# -----------------------
# HELPERS
# -----------------------
metrics = ["dp_fedavg_acc", "fedvrdp_acc", "agptr_acc", "agptr_accept_rate"]
if INCLUDE_NONDP:
    metrics += list(ROBUST_METHODS.keys())

def flatten_summary_columns(df):
    df = df.copy()
    df.columns = [
        "f" if (a == "f" and b == "") else f"{a}_{b}"
        for (a, b) in df.columns
    ]
    return df

def plot_attack_accuracy(summary_df, attack_name):
    f_pct = (summary_df["f"] * 100.0).to_numpy(dtype=float)

    def _plot_err(col_base, label):
        y = summary_df[f"{col_base}_mean"].to_numpy(dtype=float)
        yerr = summary_df[f"{col_base}_std"].to_numpy(dtype=float)
        m = np.isfinite(y)
        if m.sum() == 0:
            return
        plt.errorbar(f_pct[m], y[m], yerr=yerr[m], marker="o", capsize=3, label=label)

    plt.figure(figsize=(10, 5))
    _plot_err("dp_fedavg_acc", "DP-FedAvg")
    _plot_err("fedvrdp_acc", "FedVRDP")
    _plot_err("agptr_acc", "AG-PTR")

    if INCLUDE_NONDP:
        for col, (_, label) in ROBUST_METHODS.items():
            _plot_err(col, label)

    plt.xlabel("Byzantine fraction f (%)")
    plt.ylabel("Final test accuracy")
    plt.title(f"Experiment 11 — Robustness vs f (attack={attack_name}, ε={eps_total}, mean±std over {len(seed_list)} seeds)")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.show()

# -----------------------
# RUN ALL ATTACKS
# -----------------------
all_seedwise_rows = []
all_summary_rows = []
summary_by_attack = {}

for ATTACK in ATTACKS:
    print("\n" + "="*80)
    print(f"RUNNING FULL EXPERIMENT FOR ATTACK: {ATTACK}")
    print("="*80)

    rows = []

    for seed in seed_list:
        for f in f_list:
            acc_dp = train_dp_fedavg_robust(
                seed, eps_total, f, sigma_dp, delta=delta,
                rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
                lr0=LR0, lr_decay=LR_DECAY, momentum=MOMENTUM,
                clip_C=CLIP_C_DP,
                attack=ATTACK,
                alie_direction=ALIE_DIR, alie_z=ALIE_Z,
                sf_gamma=SF_GAMMA,
                minmax_perturb=MINMAX_PERT, diam_iters=DIAM_ITERS
            )

            acc_vr = train_fedvrdp_robust(
                seed, eps_total, f, sigma_dp, delta=delta,
                rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
                lr0=LR0, lr_decay=LR_DECAY, momentum=MOMENTUM,
                clip_C=CLIP_C_VR, k_frac=K_FRAC,
                attack=ATTACK,
                alie_direction=ALIE_DIR, alie_z=ALIE_Z,
                sf_gamma=SF_GAMMA,
                minmax_perturb=MINMAX_PERT, diam_iters=DIAM_ITERS
            )

            acc_ag, ar_ag = train_ag_ptr(
                seed, eps_total, f,
                sigma_sel, sigma_rel, delta=delta,
                rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
                lr0=LR0, lr_decay=LR_DECAY, momentum=MOMENTUM,
                rho=RHO, tau=TAU,
                R_pub_avg=R_PUB_AVG, pub_batch=PUB_BATCH, pub_scale=PUB_SCALE, public_epochs=PUBLIC_EPOCHS,
                attack=ATTACK,
                alie_direction=ALIE_DIR, alie_z=ALIE_Z,
                sf_gamma=SF_GAMMA,
                minmax_perturb=MINMAX_PERT, diam_iters=DIAM_ITERS,
                allow_zero=False
            )

            out = {
                "attack": ATTACK,
                "seed": seed,
                "f": f,
                "dp_fedavg_acc": float(acc_dp),
                "fedvrdp_acc": float(acc_vr),
                "agptr_acc": float(acc_ag),
                "agptr_accept_rate": float(ar_ag),
            }

            if INCLUDE_NONDP:
                for col, (agg, _) in ROBUST_METHODS.items():
                    out[col] = float(train_robust_baseline(
                        seed, f,
                        rounds=ROUNDS, local_epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
                        lr0=LR0, lr_decay=LR_DECAY, momentum=MOMENTUM,
                        agg=agg, attack=ATTACK,
                        clip_C=ROBUST_CLIP_C,
                        sparse_k_frac=SPARSE_K_FRAC,
                        alie_direction=ALIE_DIR, alie_z=ALIE_Z,
                        sf_gamma=SF_GAMMA,
                        minmax_perturb=MINMAX_PERT, diam_iters=DIAM_ITERS,
                        proj_k=2048
                    ))

            rows.append(out)
            all_seedwise_rows.append(out.copy())

    # per-attack seedwise
    df_seedwise_attack = pd.DataFrame(rows)
    seedwise_csv = f"exp11_{ATTACK}_seedwise.csv"
    df_seedwise_attack.to_csv(seedwise_csv, index=False)
    print("Saved", seedwise_csv)

    # per-attack summary
    summary_attack = df_seedwise_attack.groupby("f")[metrics].agg(["mean", "std"]).reset_index()
    summary_attack = flatten_summary_columns(summary_attack)
    summary_attack.insert(0, "attack", ATTACK)

    summary_csv = f"exp11_{ATTACK}_summary.csv"
    summary_attack.to_csv(summary_csv, index=False)
    print("Saved", summary_csv)

    summary_by_attack[ATTACK] = summary_attack.copy()

    # append to combined summary rows
    for _, r in summary_attack.iterrows():
        row = {"attack": ATTACK, "f": float(r["f"])}
        for metric in metrics:
            row[f"{metric}_mean"] = float(r[f"{metric}_mean"])
            row[f"{metric}_std"] = float(r[f"{metric}_std"])
        all_summary_rows.append(row)

    # plot this attack separately
    plot_attack_accuracy(summary_attack, ATTACK)

# -----------------------
# COMBINED TABLES
# -----------------------
df_all_seedwise = pd.DataFrame(all_seedwise_rows)
df_all_seedwise.to_csv("exp11_all_attacks_seedwise.csv", index=False)
print("Saved exp11_all_attacks_seedwise.csv")

df_all_summary = pd.DataFrame(all_summary_rows)
df_all_summary = df_all_summary.sort_values(["attack", "f"]).reset_index(drop=True)
df_all_summary.to_csv("exp11_all_attacks_summary.csv", index=False)
print("Saved exp11_all_attacks_summary.csv")

print("\n===== COMBINED SUMMARY TABLE (ALL attacks, mean±std over seeds) =====")
display(df_all_summary)

# Optional compact accuracy-only table
acc_cols = ["attack", "f", "dp_fedavg_acc_mean", "fedvrdp_acc_mean", "agptr_acc_mean"]
if INCLUDE_NONDP:
    acc_cols += [
        "median_nondp_acc_mean",
        "trimmean_nondp_acc_mean",
        "krum_nondp_acc_mean",
        "sparsefed_nondp_acc_mean",
    ]

df_all_accuracy_only = df_all_summary[acc_cols].copy()
print("\n===== COMBINED ACCURACY TABLE (means only) =====")
display(df_all_accuracy_only)

print("NOTE: Krum may show NaN at high f (invalid regime for m=100). That is expected.")

FULL Exp11 settings
  attacks: ['ALIE', 'SF', 'MINMAX', 'MINSUM']
  eps_total: 2.0 delta: 1e-05
  rounds: 180 local_epochs: 10 batch: 10
  DP sigma: 0.9723217465539842 achieved eps≈ 1.9999999999999973
  AG sigmas sel/rel: 1.965902445173369 0.9829512225866845 achieved eps≈ 1.999999999999998
  INCLUDE_NONDP: True

RUNNING FULL EXPERIMENT FOR ATTACK: ALIE


SPARSEFED(non-DP) atk=ALIE f=20%:  63%|██████▎   | 113/180 [03:37<02:09,  1.93s/it]


KeyboardInterrupt: 

In [16]:
# Cell 15A — Verify ALIE files + save partial in-memory progress after interrupt

import os
import pandas as pd

print("Checking saved ALIE files...")
print("exp11_ALIE_seedwise.csv exists:", os.path.exists("exp11_ALIE_seedwise.csv"))
print("exp11_ALIE_summary.csv exists:", os.path.exists("exp11_ALIE_summary.csv"))

if os.path.exists("exp11_ALIE_seedwise.csv"):
    df_alie_saved = pd.read_csv("exp11_ALIE_seedwise.csv")
    print("ALIE saved rows:", len(df_alie_saved))
    display(df_alie_saved.head())
else:
    print("ALIE seedwise file not found on disk yet.")

# Save whatever is already in memory from the interrupted run
if "all_seedwise_rows" in globals() and len(all_seedwise_rows) > 0:
    df_partial = pd.DataFrame(all_seedwise_rows)
    if all(c in df_partial.columns for c in ["attack", "seed", "f"]):
        df_partial = df_partial.drop_duplicates(subset=["attack", "seed", "f"], keep="last").reset_index(drop=True)
    df_partial.to_csv("exp11_partial_after_interrupt.csv", index=False)
    print("Saved partial checkpoint: exp11_partial_after_interrupt.csv")
    print("Partial rows:", len(df_partial))
    display(df_partial.tail())
else:
    print("No in-memory partial rows found. That's okay if ALIE is already saved on disk.")

Checking saved ALIE files...
exp11_ALIE_seedwise.csv exists: False
exp11_ALIE_summary.csv exists: False
ALIE seedwise file not found on disk yet.
Saved partial checkpoint: exp11_partial_after_interrupt.csv
Partial rows: 16


,attack,seed,f,dp_fedavg_acc,fedvrdp_acc,agptr_acc,agptr_accept_rate,median_nondp_acc,trimmean_nondp_acc,krum_nondp_acc,sparsefed_nondp_acc
11,ALIE,1,0.40,0.2326,0.2685,0.7338,0.150000,0.4239,0.2015,0.100,0.2568
12,ALIE,1,0.49,0.1918,0.1092,0.6269,0.033333,0.0992,0.1037,NaN,0.2907
13,ALIE,1,0.60,0.1809,0.0981,0.1058,0.005556,0.1000,0.1000,NaN,0.3677
14,ALIE,2,0.00,0.7994,0.8165,0.7730,0.727778,0.8058,0.8604,0.750,0.8576
15,ALIE,2,0.10,0.7589,0.7851,0.7548,0.738889,0.7890,0.7925,0.718,0.8065
